In [2]:
import os
import json
import time
from tqdm import tqdm
from openai import OpenAI

# ============================================================
# 0. 配置区
#    你主要只需要改 WORKER_ID
# ============================================================

API_KEY = "sk-yqcegwvpdomtkrpnopynmdehgenmpzenjlaftxgxbcibqegv"
BASE_URL = "https://api.siliconflow.cn/v1" 
MODEL_NAME = "Pro/deepseek-ai/DeepSeek-V3.2"

# 输入文件
INPUT_PATH = "../fit_dualsg_all.json"

# 输出目录
OUTPUT_DIR = "../llm_batch_outputs"

# 每个worker处理多少条
CHUNK_SIZE = 3000

# 当前worker编号，只改这个
# 例如：
# WORKER_ID = 0 -> [0, 3000)
# WORKER_ID = 1 -> [3000, 6000)
# ...
WORKER_ID = 0

# 每批给模型多少条
BATCH_SIZE = 10

# 最大重试次数
MAX_RETRIES = 5

# 是否断点续跑
RESUME = True

# 失败后重试基础等待时间
RETRY_SLEEP = 2

# 每次成功请求后休息一下，避免太快
REQUEST_SLEEP = 0.2


# ============================================================
# 1. 自动计算当前worker的处理范围和输出文件名
# ============================================================

START_IDX = WORKER_ID * CHUNK_SIZE
END_IDX = START_IDX + CHUNK_SIZE

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    f"fit_dualsg_llm_batch10_worker{WORKER_ID}_{START_IDX}_{END_IDX}.jsonl"
)

ERROR_PATH = os.path.join(
    OUTPUT_DIR,
    f"fit_dualsg_llm_batch10_worker{WORKER_ID}_{START_IDX}_{END_IDX}_error.jsonl"
)

print("当前WORKER_ID:", WORKER_ID)
print("处理范围:", START_IDX, END_IDX)
print("输出文件:", OUTPUT_PATH)
print("错误文件:", ERROR_PATH)


# ============================================================
# 2. 初始化客户端
# ============================================================

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)


# ============================================================
# 3. Prompt
# ============================================================

SYSTEM_PROMPT = """
You are a time series analysis expert.

You will receive multiple time series samples.

For each sample, generate a semantic description with EXACTLY 5 lines:

Overall trend: ...
Volatility: ...
Turning points: ...
Temporal pattern: ...
Ending behavior: ...

Return ONLY valid JSON in the following format:

[
  {"id": 0, "annotation": "..."},
  {"id": 1, "annotation": "..."}
]

Rules:
- id must match input order (0 to N-1)
- no explanation
- no markdown
- no extra text
"""


# ============================================================
# 4. 工具函数
# ============================================================

def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def load_done_indices(path):
    """
    从已有输出文件中读取已经完成的index
    用于断点续跑
    """
    done = set()
    if not os.path.exists(path):
        return done

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                idx = obj.get("index", None)
                if isinstance(idx, int):
                    done.add(idx)
            except:
                continue
    return done


def build_batch_prompt(batch_series):
    """
    构造一批10条的输入
    """
    text = "Analyze the following time series samples:\n\n"
    for i, s in enumerate(batch_series):
        text += f"Sample {i}: {json.dumps(s, ensure_ascii=False)}\n"
    return text


def clean_json_text(text):
    """
    清理模型可能返回的markdown包裹
    """
    text = text.replace("```json", "").replace("```", "").strip()
    return text


def validate_annotation_text(annotation):
    """
    校验单条annotation是否是5行结构
    """
    if not isinstance(annotation, str):
        return False

    lines = [line.strip() for line in annotation.strip().split("\n") if line.strip()]
    if len(lines) != 5:
        return False

    prefixes = [
        "Overall trend:",
        "Volatility:",
        "Turning points:",
        "Temporal pattern:",
        "Ending behavior:"
    ]

    for line, prefix in zip(lines, prefixes):
        if not line.startswith(prefix):
            return False

    return True


def validate_batch_output(result, batch_size):
    """
    校验整批输出是否合格
    要求：
    1. 是list
    2. 长度等于batch_size
    3. id必须是0~batch_size-1
    4. 每条annotation合法
    """
    if not isinstance(result, list):
        return False

    if len(result) != batch_size:
        return False

    ids = []
    for item in result:
        if not isinstance(item, dict):
            return False
        if "id" not in item or "annotation" not in item:
            return False
        ids.append(item["id"])
        if not validate_annotation_text(item["annotation"]):
            return False

    if sorted(ids) != list(range(batch_size)):
        return False

    return True


# ============================================================
# 5. 读取数据
# ============================================================

data = load_data(INPUT_PATH)
TOTAL_N = len(data)

# 自动截断，避免越界
REAL_END_IDX = min(END_IDX, TOTAL_N)

print("总样本数:", TOTAL_N)
print("当前worker实际处理范围:", START_IDX, REAL_END_IDX)

if START_IDX >= TOTAL_N:
    print("当前WORKER_ID超出数据范围，无需处理。")
    raise SystemExit


# ============================================================
# 6. 断点续跑：读取已完成的index
# ============================================================

done_indices = set()
if RESUME:
    done_indices = load_done_indices(OUTPUT_PATH)
    print("已完成数量:", len(done_indices))

all_indices = list(range(START_IDX, REAL_END_IDX))
todo_indices = [i for i in all_indices if i not in done_indices]

print("待处理数量:", len(todo_indices))


# ============================================================
# 7. 按10条分组
# ============================================================

batches = [
    todo_indices[i:i + BATCH_SIZE]
    for i in range(0, len(todo_indices), BATCH_SIZE)
]

print("总batch数:", len(batches))


# ============================================================
# 8. 主循环
# ============================================================

for batch in tqdm(batches, desc=f"Worker {WORKER_ID} 进度"):

    # 最后一组不足10条时，这里也照样处理
    current_batch_size = len(batch)

    batch_series = []
    batch_valid = True

    for idx in batch:
        item = data[idx]
        series = item.get("series", None)

        # series必须是非空list
        if not isinstance(series, list) or len(series) == 0:
            batch_valid = False
            break

        batch_series.append(series)

    # 如果这一批数据本身有问题，整批记失败
    if not batch_valid:
        for idx in batch:
            fail_obj = {
                "index": idx,
                "status": "failed",
                "annotation": None,
                "error": "当前batch中存在非法series"
            }
            write_jsonl(OUTPUT_PATH, fail_obj)
            write_jsonl(ERROR_PATH, fail_obj)
        continue

    prompt = build_batch_prompt(batch_series)

    success = False
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ]
            )

            text = response.output_text
            text = clean_json_text(text)

            result = json.loads(text)

            # 用当前批真实大小校验
            if not validate_batch_output(result, current_batch_size):
                raise ValueError("JSON结构或annotation格式不符合要求")

            # 整批成功，写入输出
            for item in result:
                local_id = item["id"]
                global_idx = batch[local_id]

                out_obj = {
                    "index": global_idx,
                    "status": "ok",
                    "annotation": item["annotation"]
                }
                write_jsonl(OUTPUT_PATH, out_obj)

            success = True
            break

        except Exception as e:
            last_error = f"{type(e).__name__}: {str(e)}"
            time.sleep(RETRY_SLEEP * attempt)

    # 如果整批最终失败，则整批都记失败
    if not success:
        for idx in batch:
            fail_obj = {
                "index": idx,
                "status": "failed",
                "annotation": None,
                "error": last_error
            }
            write_jsonl(OUTPUT_PATH, fail_obj)
            write_jsonl(ERROR_PATH, fail_obj)

    if REQUEST_SLEEP > 0:
        time.sleep(REQUEST_SLEEP)

print("当前worker处理完成。")
print("输出文件:", OUTPUT_PATH)
print("错误文件:", ERROR_PATH)

当前WORKER_ID: 0
处理范围: 0 3000
输出文件: ../llm_batch_outputs/fit_dualsg_llm_batch10_worker0_0_3000.jsonl
错误文件: ../llm_batch_outputs/fit_dualsg_llm_batch10_worker0_0_3000_error.jsonl
总样本数: 431592
当前worker实际处理范围: 0 3000
已完成数量: 10
待处理数量: 2990
总batch数: 299


Worker 0 进度:   2%|▏         | 5/299 [35:06<34:24:34, 421.34s/it]


KeyboardInterrupt: 